# Land cover class labels and colors CSV

This notebook makes `io-lulc-9-class-labels.csv`, the CSV with the class codes, names and official colors of the [Impact Observatory 10 m Annual Land Use Land Cover (9-class) V2 maps](https://planetarycomputer.microsoft.com/dataset/io-lulc-annual-v02) used in the land cover statistics lesson.

- The class codes and names come from the STAC item's metadata.
- The colors come from the Microsoft Planetary Computer (MPC) [class map for `io-lulc-9-class`](https://planetarycomputer.microsoft.com/api/data/v1/legend/classmap/io-lulc-9-class), converted from RGBA to hex codes. No Data is fully transparent in the class map, so it gets white.

The CSV is saved to `data/` and should be uploaded to the course's Google Drive.

In [1]:
import os

import pandas as pd
import requests
from matplotlib.colors import to_hex

import pystac_client
import planetary_computer

## Class codes and names from the item's metadata

All the items in the collection share the same classes, so any item works. We use the 2023 map, as in the lesson:

In [2]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Any point within the collection's coverage works; this one is in the Eaton Fire area
search = catalog.search(collections=['io-lulc-annual-v02'],
                        intersects={'type': 'Point', 'coordinates': [-118.1, 34.2]},
                        datetime='2023')
item = search.item_collection()[0]
assert item.properties['start_datetime'].startswith('2023')

class_values = item.assets['data'].extra_fields['file:values']

labels = pd.DataFrame({'code': [c['values'][0] for c in class_values],
                       'class_label': [c['summary'] for c in class_values]})
labels

,code,class_label
0,0,No Data
1,1,Water
2,2,Trees
3,4,Flooded vegetation
4,5,Crops
5,7,Built area
6,8,Bare ground
7,9,Snow/ice
8,10,Clouds
9,11,Rangeland


## Official colors from the MPC class map

The class map has an RGBA value (0-255) for each code, including codes 3 and 6, which this product doesn't use. We only keep the codes in the metadata:

In [3]:
classmap = requests.get(
    'https://planetarycomputer.microsoft.com/api/data/v1/legend/classmap/io-lulc-9-class'
    ).json()
classmap

{'0': [0, 0, 0, 0],
 '1': [65, 155, 223, 255],
 '2': [57, 125, 73, 255],
 '3': [136, 176, 83, 0],
 '4': [122, 135, 198, 255],
 '5': [228, 150, 53, 255],
 '6': [223, 195, 90, 0],
 '7': [196, 40, 27, 255],
 '8': [165, 155, 143, 255],
 '9': [168, 235, 255, 255],
 '10': [97, 97, 97, 255],
 '11': [227, 226, 195, 255]}

In [4]:
def rgba_to_hex(rgba):
    """Convert an RGBA list (0-255) to a hex code, using white for fully transparent colors."""
    if rgba[3] == 0:
        return '#ffffff'
    return to_hex([value / 255 for value in rgba[:3]])

labels['color'] = [rgba_to_hex(classmap[str(code)]) for code in labels['code']]
labels

,code,class_label,color
0,0,No Data,#ffffff
1,1,Water,#419bdf
2,2,Trees,#397d49
3,4,Flooded vegetation,#7a87c6
4,5,Crops,#e49635
5,7,Built area,#c4281b
6,8,Bare ground,#a59b8f
7,9,Snow/ice,#a8ebff
8,10,Clouds,#616161
9,11,Rangeland,#e3e2c3


## Save the CSV

In [5]:
labels.to_csv(os.path.join('data', 'io-lulc-9-class-labels.csv'), index=False)

# Check it reads back the same
assert pd.read_csv(os.path.join('data', 'io-lulc-9-class-labels.csv')).equals(labels)